# Eksperimen Tugas Akhir: Rancang Bangun Clinical Decision Support System Deteksi Kanker Menggunakan Seleksi Fitur Dimensi Tinggi (SVM-RFE) dan Optimasi Hyperparameter untuk Meminimalkan False Negative

**Penyusun:** Nayyara Rauf Ghani Al Khayri (NIM: A11.2024.16055)  
**Dosen Pembimbing:** Abu Salam, M.Kom  
**Program Studi:** S1 Teknik Informatika - Peminatan Data Mining & Bio-Informatika  
**Landasan Riset Utama:** Isabelle Guyon, Jason Weston, Stephen Barnhill, & Vladimir Vapnik (2002) - *"Gene Selection for Cancer Classification using Support Vector Machines"*, Machine Learning, 46, 389–422.

---

## 1. Abstrak & Latar Belakang Klinis

Analisis profil ekspresi gen mikroarray (*gene expression microarray*) dan biomarker morfologis seluler pada sistem pendukung keputusan klinis (*Clinical Decision Support System* / CDSS) onkologi berhadapan langsung dengan fenomena **Curse of Dimensionality** (Bellman, 1961), di mana jumlah fitur genomik ($P$) sering kali melampaui atau sebanding dengan jumlah observasi pasien ($N$), yakni $P \gg N$ atau $P \approx N$.

Dalam domain onkologi diagnostik medis, kesalahan klasifikasi memiliki konsekuensi klinis yang sangat **asimetris**:
$$\text{Cost}(\text{False Negative}) \gg \text{Cost}(\text{False Positive})$$
* **False Negative (FN) - Pasien Kanker Malignant Didiagnosis Jinak (Benign):** Merupakan kegagalan klinis fatal. Pasien kehilangan jendela emas intervensi kuratif (seperti mastektomi, kemoterapi lini pertama, atau radioterapi), memicu metastasis sel kanker yang mengancam keselamatan jiwa.
* **False Positive (FP) - Pasien Jinak (Benign) Didiagnosis Kanker (Malignant):** Mengakibatkan pemeriksaan biopsi jaringan konfirmasi atau mamografi ulang. Meskipun menimbulkan beban psikologis dan biaya prosedural sementara, keselamatan jiwa pasien tetap terjaga.

Oleh sebab itu, tujuan utama perancangan sistem CDSS ini adalah **memaksimalkan Sensitivitas (Recall)** dan **F2-Score** serta mereduksi jumlah pasien *False Negative* hingga seminimal mungkin. Untuk mengatasi jebakan kolinearitas dan gen derau (*noise genes*) pada data dimensi tinggi, eksperimen komprehensif ini membandingkan metode filter korelasi univariat klasik **Golub et al. (1999)** terhadap algoritma seleksi fitur multivariat rekursif **SVM-RFE Guyon et al. (2002)**, yang selanjutnya dioptimasi menggunakan **Bayesian Hyperparameter Optimization (Optuna)** dan kalibrasi ambang batas (*threshold calibration*).

---

## 2. Landasan Teori & Formulasi Matematis

### 2.1. Korelasi Univariat Golub et al. (1999) - Persamaan (2) Guyon et al. (2002)
Todd Golub et al. (1999) dalam riset seminal klasifikasi leukemia mikroarray mendefinisikan metrik pemeringkat korelasi univariat sinyal-ke-derau (*Signal-to-Noise Ratio* / SNR):

$$w_i^{\text{Golub}} = \frac{\mu_i(+) - \mu_i(-)}{\sigma_i(+) + \sigma_i(-)}$$

Keterangan:
* $\mu_i(+)$ dan $\sigma_i(+)$ : Rata-rata (*mean*) dan standar deviasi fitur ke-$i$ pada populasi sampel kelas positif ($y = +1$, Malignant / Kanker).
* $\mu_i(-)$ dan $\sigma_i(-)$ : Rata-rata (*mean*) dan standar deviasi fitur ke-$i$ pada populasi sampel kelas negatif ($y = 0$, Benign / Jinak).
* Kriteria ranking: Fitur diurutkan secara menurun berdasarkan nilai magnitudo mutlak korelasi: $|w_i^{\text{Golub}}|$.
* **Keterbatasan Kritis Teoretis:** Metode Golub mengevaluasi setiap fitur secara terisolasi (univariat). Jika terdapat kelompok gen yang mengalami ko-ekspresi serentak (*collinear biological pathways*), metode ini akan memilih seluruh salinan gen redundan tersebut sekaligus, sehingga menghabiskan kuota fitur dan menyingkirkan biomarker independen yang krusial.

### 2.2. Algoritma SVM-RFE & Kriteria Ranking OBD Guyon et al. (2002)
Isabelle Guyon et al. (2002) merumuskan **Support Vector Machine - Recursive Feature Elimination (SVM-RFE)** berbasis aproksimasi *Optimal Brain Damage* (OBD; LeCun, Denker, & Solla, 1990).

Diberikan fungsi pemisah hyperplane Linear Support Vector Machine:
$$f(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b = \sum_{k=1}^N \alpha_k y_k (\mathbf{x}_k \cdot \mathbf{x}) + b$$
dengan fungsi objektif margin primal:
$$J = \frac{1}{2} \|\mathbf{w}\|^2 = \frac{1}{2} \sum_{i=1}^P w_i^2$$

Ketika fitur ke-$i$ dieliminasi dari representasi data, perubahan variasi fungsi biaya diaproksimasi melalui ekspansi deret Taylor orde kedua:
$$DJ(i) = \frac{1}{2} \frac{\partial^2 J}{\partial w_i^2} (Dw_i)^2 = \frac{1}{2} (w_i)^2$$

Kriteria pemeringkatan fitur (*Ranking Criterion*) Guyon et al. (2002) didefinisikan sebagai kuadrat bobot koefisien pemisah:
$$c_i = (w_i)^2$$

Fitur dengan skor $c_i$ terkecil adalah fitur yang memberikan kontribusi paling minimal terhadap lebar margin pemisah, sehingga dieliminasi secara rekursif (*backward elimination*) pada setiap iterasi.

### 2.3. Formulasi Sensitivitas / Recall Klinis dan Asimetri Biaya
Dalam matriks konfusi diagnostik onkologi ($N_{\text{total}} = \text{TP} + \text{TN} + \text{FP} + \text{FN}$), metrik kinerja didefinisikan sebagai:

$$\text{Recall (Sensitivity)} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

$$\text{Specificity} = \frac{\text{TN}}{\text{TN} + \text{FP}}$$

$$\text{Precision (PPV)} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

$$F_2\text{-Score} = (1 + 2^2) \frac{\text{Precision} \cdot \text{Recall}}{2^2 \cdot \text{Precision} + \text{Recall}} = \frac{5 \cdot \text{Precision} \cdot \text{Recall}}{4 \cdot \text{Precision} + \text{Recall}}$$

Parameter $\beta = 2$ pada $F_2$-Score memberikan bobot penalti Recall 4 kali lebih besar daripada Precision ($2^2 = 4$), menjadikannya fungsi objektif yang ideal dalam optimasi CDSS deteksi dini kanker.

In [ ]:
import sys
from pathlib import Path
import warnings

# Menghindari peringatan deprecation dan formatting
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Menambahkan direktori root proyek ke sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, fbeta_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix
)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from IPython.display import display

# Import dependensi modular dari paket src/
from src.config import (
    RAW_DATA_PATH, PROCESSED_DATA_PATH, MODEL_ARTIFACT_PATH,
    METADATA_PATH, RANDOM_STATE, DEFAULT_N_FEATURES_TO_SELECT,
    RECALL_OPTIMIZED_THRESHOLD
)
from src.data_loader import (
    generate_high_dimensional_medical_data,
    guyon_preprocessing,
    get_train_test_data
)
from src.feature_selection import GolubBaselineSelector, SVMRFEGuyonSelector
from src.optimizer import (
    compute_guyon_margins,
    optimize_svm_hyperparameters,
    evaluate_clinical_metrics
)

# Konfigurasi Estetika Visualisasi Publikasi Ilmiah (Kompatibel Dark & Light Theme)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 110,
    "axes.edgecolor": "#cccccc",
    "axes.linewidth": 1.0,
    "grid.color": "#e0e0e0",
    "grid.linestyle": "--",
    "grid.alpha": 0.7
})

print(f"[OK] Setup environment sukses. Direktori root proyek: {ROOT_DIR}")
print("[OK] Modul src (config, data_loader, feature_selection, optimizer) berhasil diimpor.")

## 3. Data Ingestion & Audit Dimensi Tinggi

### Arsitektur Data Mikroarray Simulasi
Untuk merefleksikan kondisi riil analisis data genomik dan patologi onkologi, dataset medis dirancang dengan menggabungkan dua domain:
1. **30 Biomarker Klinis Riil ($P_1 = 30$):** Berasal dari data *Breast Cancer Wisconsin Diagnostic (WDBC)* hasil biopsi aspirasi jarum halus (*Fine Needle Aspirate* / FNA). Mengukur 10 karakteristik geometris nukleus sel (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension) dalam 3 estimasi: mean, standard error (SE), dan worst (ekstrem terbesar).
2. **15 Gen Kolinear Redundan ($P_2 = 15$):** Mengimitasi fenomena biologis ko-ekspresi genetik sesuai paper Guyon et al. (2002) Bagian 6.2.1. Gen-gen ini dibangkitkan dari biomarker `mean radius` terstandarisasi dengan penambahan perturbasi derau Gaussian kecil:
   $$g_j = \tilde{x}_{\text{radius}} + \epsilon_j, \quad \epsilon_j \sim \mathcal{N}(0, 0.25^2), \quad j \in [1, 15]$$
3. **55 Gen Gaussian Pure Noise ($P_3 = 55$):** Menggambarkan fluktuasi latar instrumen mikroarray tanpa asosiasi biologis terhadap keganasan tumor:
   $$g_k \sim \mathcal{N}(0, 1), \quad k \in [16, 70]$$

Total ruang fitur pengamatan adalah $P = 100$ fitur dengan $N = 569$ observasi pasien. Target biner didefinisikan sebagai:
* $y = 1$ : **Malignant** (Keganasan / Kanker) — Kelas Positif Prioritas Tinggi
* $y = 0$ : **Benign** (Jinak / Non-kanker) — Kelas Negatif

In [ ]:
# 1. Ingestion dataset gabungan (30 biomarker klinis WDBC + 70 gen sintetis)
df_raw = generate_high_dimensional_medical_data(n_noise_genes=70)
X_train_raw, X_test_raw, y_train, y_test, all_features = get_train_test_data(test_size=0.2, n_noise_genes=70)

N_total, total_cols = df_raw.shape
total_features = total_cols - 1  # kurangi kolom target_malignant
clinical_features = all_features[:30]
collinear_features = all_features[30:45]
noise_features = all_features[45:]

n_benign = int(np.sum(df_raw["target_malignant"] == 0))
n_malignant = int(np.sum(df_raw["target_malignant"] == 1))

# 2. Ringkasan Audit Dimensi Matriks Data
audit_df = pd.DataFrame({
    "Parameter Matriks": [
        "Total Sampel Pasien (N)",
        "Sampel Data Latih (Train Set, 80%)",
        "Sampel Data Uji (Test Set, 20%)",
        "Total Fitur Pengamatan (P)",
        "Fitur Biomarker Klinis Asli (WDBC FNA)",
        "Fitur Gen Kolinear Redundan (Guyon 6.2.1)",
        "Fitur Gen Pure Gaussian Noise Mikroarray",
        "Rasio Observasi-terhadap-Fitur (N / P)",
        "Pasien Diagnosis Jinak / Benign (y = 0)",
        "Pasien Diagnosis Kanker / Malignant (y = 1)",
        "Prevalensi Kasus Kanker Positif"
    ],
    "Nilai Statistik": [
        f"{N_total} pasien",
        f"{len(y_train)} pasien ({len(y_train)/N_total*100:.1f}%)",
        f"{len(y_test)} pasien ({len(y_test)/N_total*100:.1f}%)",
        f"{total_features} fitur",
        f"{len(clinical_features)} fitur (30.0%)",
        f"{len(collinear_features)} fitur (15.0%)",
        f"{len(noise_features)} fitur (55.0%)",
        f"{N_total / total_features:.2f} (Sparsitas Tinggi)",
        f"{n_benign} pasien ({n_benign / N_total * 100:.1f}%)",
        f"{n_malignant} pasien ({n_malignant / N_total * 100:.1f}%)",
        f"{n_malignant / N_total * 100:.2f}%"
    ]
})

print("=== RINGKASAN AUDIT DIMENSI MATRIKS DATA MEDIS ===")
display(audit_df)

# 3. Visualisasi Distribusi Kelas Target & Heatmap Korelasi Matriks
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Distribusi Kelas Target
palette_classes = ["#2a9d8f", "#e63946"]
counts = [n_benign, n_malignant]
labels = [f"Benign / Jinak\n({n_benign} pasien)", f"Malignant / Kanker\n({n_malignant} pasien)"]
bars = axes[0].bar(["Benign (y=0)", "Malignant (y=1)"], counts, color=palette_classes, width=0.55, edgecolor="black", linewidth=1.2)
axes[0].set_title("Distribusi Rasio Pasien (Benign vs Malignant)", fontweight="bold")
axes[0].set_ylabel("Jumlah Pasien")
axes[0].set_ylim(0, max(counts) * 1.18)
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{yval} ({yval/N_total*100:.1f}%)", ha="center", va="bottom", fontweight="bold")

# Subplot 2: Heatmap Korelasi Matriks (Biomarker Klinis vs Gen Kolinear vs Gen Noise)
sample_subset_cols = clinical_features[:6] + collinear_features[:6] + noise_features[:4]
corr_matrix = df_raw[sample_subset_cols].corr()

sns.heatmap(
    corr_matrix,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    annot=False,
    cbar_kws={"label": "Koefisien Korelasi Pearson (r)"},
    ax=axes[1]
)
axes[1].set_title("Heatmap Korelasi Submatriks Fitur\n(Klinis Asli vs Kolinear Redundan vs Pure Noise)", fontweight="bold")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 4. Preprocessing Standar Guyon et al. (2002)

### Landasan Matematis Standardisasi & Arctan Squashing
Dalam paper Guyon et al. (2002) Bagian 5.1 & 5.2, dua tahapan transformasi data wajib dilakukan sebelum seleksi fitur berbasis margin:

1. **Standardisasi Z-Score (Zero-Mean, Unit-Variance):**
   Mencegah fitur dengan rentang nilai absolut besar (misal area lesi dalam skala ribuan) mendominasi gradien fungsi margin:
   $$\tilde{x}_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$
   Parameter $\mu_j$ dan $\sigma_j$ diestimasi secara ketat hanya dari data latih (*training split*) lalu diterapkan ke data uji untuk mencegah kebocoran informasi (*data leakage*).

2. **Fungsi Non-Linear Squashing (Peredam Outlier Ekstrem):**
   Pengukuran klinis onkologi kerap memuat observasi asimetris berat (*heavy-tailed outliers*). Guyon et al. (2002) mengusulkan fungsi saturasi arctan:
   $$f(\tilde{x}) = c \cdot \arctan\left(\frac{\tilde{x}}{c}\right)$$
   dengan parameter saturasi $c = 2.0$.
   
   * **Perilaku Linear di Pusat Distribusi ($|\tilde{x}| \ll c$):** Mengingat $\arctan(u) \approx u$ untuk $u \approx 0$, maka $f(\tilde{x}) \approx \tilde{x}$. Informasi variabilitas normal pasien di sekitar median tetap terjaga utuh.
   * **Batas Asimtotik pada Outlier ($|\tilde{x}| \gg c$):** Nilai dibatasi secara halus pada nilai konstan:
     $$\lim_{\tilde{x} \to \pm\infty} f(\tilde{x}) = \pm c \cdot \frac{\pi}{2} \approx \pm 2.0 \cdot 1.5708 \approx \pm 3.1416$$
   Hal ini mencegah sampel ekstrem menarik posisi bidang pemisah (*decision boundary*) SVM secara berlebihan.

In [ ]:
# 1. Eksekusi Preprocessing Guyon et al. (2002)
# Standardisasi Mean-Std + Fungsi Squashing Arctan (c = 2.0)
X_train_std = (X_train_raw - np.mean(X_train_raw, axis=0)) / (np.std(X_train_raw, axis=0) + 1e-8)
X_train_guyon, X_test_guyon, mean_vec, std_vec = guyon_preprocessing(
    X_train_raw, X_test_raw, apply_squashing=True, c=2.0
)

# Indeks fitur berat outlier untuk audit statistik
idx_area_err = all_features.index("area error")
idx_worst_area = all_features.index("worst area")
idx_worst_perim = all_features.index("worst perimeter")

features_to_audit = [("area error", idx_area_err), ("worst area", idx_worst_area), ("worst perimeter", idx_worst_perim)]

stats_rows = []
for name, idx in features_to_audit:
    raw_col = X_train_raw[:, idx]
    squashed_col = X_train_guyon[:, idx]
    stats_rows.append({
        "Nama Fitur": name,
        "Raw Skewness": f"{skew(raw_col):.2f}",
        "Squashed Skewness": f"{skew(squashed_col):.2f}",
        "Raw Kurtosis": f"{kurtosis(raw_col):.2f}",
        "Squashed Kurtosis": f"{kurtosis(squashed_col):.2f}",
        "Raw Range [Min, Max]": f"[{raw_col.min():.1f}, {raw_col.max():.1f}]",
        "Squashed Range [Min, Max]": f"[{squashed_col.min():.2f}, {squashed_col.max():.2f}]"
    })

stats_df = pd.DataFrame(stats_rows)
print("=== PERBANDINGAN STATISTIK DESKRIPTIF SEBELUM & SESUDAH GUYON SQUASHING ===")
display(stats_df)

# 2. Visualisasi Kurva Transfer Teoretis & Plot KDE Distribusi
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: Kurva Teoretis Arctan Squashing f(x) = c * arctan(x / c)
x_vals = np.linspace(-10, 10, 500)
c_const = 2.0
y_squashed = c_const * np.arctan(x_vals / c_const)

axes[0].plot(x_vals, x_vals, "--", color="gray", lw=1.5, label="Fungsi Identitas Linear f(x) = x")
axes[0].plot(x_vals, y_squashed, "-", color="#d62728", lw=2.5, label=f"Guyon Arctan f(x) = {c_const} arctan(x/{c_const})")
axes[0].axhline(c_const * np.pi / 2, color="teal", ls=":", lw=1.8, label=f"Batas Atas (+{c_const*np.pi/2:.2f})")
axes[0].axhline(-c_const * np.pi / 2, color="teal", ls=":", lw=1.8, label=f"Batas Bawah (-{c_const*np.pi/2:.2f})")
axes[0].set_title("Karakteristik Teoretis Squashing Arctan", fontweight="bold")
axes[0].set_xlabel("Z-Score Input (x)")
axes[0].set_ylabel("Nilai Output f(x)")
axes[0].set_ylim(-6, 6)
axes[0].legend(loc="upper left")

# Subplot 2: Distribusi KDE 'area error'
sns.kdeplot(X_train_std[:, idx_area_err], color="#1f77b4", lw=2, fill=True, alpha=0.3, ax=axes[1], label=f"Standardized (Skew: {skew(X_train_std[:, idx_area_err]):.2f})")
sns.kdeplot(X_train_guyon[:, idx_area_err], color="#2ca02c", lw=2, fill=True, alpha=0.4, ax=axes[1], label=f"Guyon Squashed (Skew: {skew(X_train_guyon[:, idx_area_err]):.2f})")
axes[1].set_title("KDE Distribusi: area error", fontweight="bold")
axes[1].set_xlabel("Nilai Fitur")
axes[1].set_ylabel("Densitas")
axes[1].legend()

# Subplot 3: Distribusi KDE 'worst area'
sns.kdeplot(X_train_std[:, idx_worst_area], color="#ff7f0e", lw=2, fill=True, alpha=0.3, ax=axes[2], label=f"Standardized (Skew: {skew(X_train_std[:, idx_worst_area]):.2f})")
sns.kdeplot(X_train_guyon[:, idx_worst_area], color="#2ca02c", lw=2, fill=True, alpha=0.4, ax=axes[2], label=f"Guyon Squashed (Skew: {skew(X_train_guyon[:, idx_worst_area]):.2f})")
axes[2].set_title("KDE Distribusi: worst area", fontweight="bold")
axes[2].set_xlabel("Nilai Fitur")
axes[2].set_ylabel("Densitas")
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Eksperimen Seleksi Fitur Komparatif: Golub (1999) vs SVM-RFE Guyon (2002)

### Pertanyaan Ilmiah Eksperimen:
1. **Apakah Pemeringkat Univariat Golub (1999) Terjebak Kolinearitas?**
   Metode Golub mengukur skor korelasi secara univariat tanpa mempertimbangkan ketergantungan antar-fitur. Ke-15 gen sintetis kolinear yang diturunkan dari `mean radius` memiliki korelasi tinggi terhadap target kanker sehingga berpotensi membajak (*hijack*) kuota seleksi fitur.
2. **Apakah SVM-RFE Guyon et al. (2002) Mampu Memilih Fitur Komplementer?**
   SVM-RFE mengevaluasi vektor bobot hyperplane $\mathbf{w}$ secara multivariat. Ketika satu fitur ukuran sel telah terpilih, bobot fitur-fitur redundan akan terkoreksi mengecil sehingga fitur komplementer (seperti tekstur atau simetri nukleus) yang memiliki kontribusi unik dapat dipertahankan.

Kedua algoritma ditugaskan memilih $d = 12$ fitur optimal dari 100 fitur pengamatan.

In [ ]:
# 1. Eksekusi GolubBaselineSelector vs SVMRFEGuyonSelector untuk memilih 12 fitur terbaik
n_select = DEFAULT_N_FEATURES_TO_SELECT

golub_sel = GolubBaselineSelector(n_features_to_select=n_select).fit(X_train_guyon, y_train)
golub_features = [all_features[i] for i in golub_sel.selected_indices_]

rfe_sel = SVMRFEGuyonSelector(n_features_to_select=n_select, C=0.1, step=1).fit(X_train_guyon, y_train)
rfe_features = [all_features[i] for i in rfe_sel.selected_indices_]
rfe_weights_sq = rfe_sel.final_weights_

# 2. Pengecekan Fitur: Berapa banyak fitur klinis asli terpilih vs fitur noise
def categorize_feature(fname: str) -> str:
    if fname.startswith("GENE_EXPR_"):
        gene_idx = int(fname.split("_")[-1])
        return "Gen Kolinear Redundan" if gene_idx <= 15 else "Gen Pure Noise"
    return "Biomarker Klinis Asli"

golub_categories = [categorize_feature(f) for f in golub_features]
rfe_categories = [categorize_feature(f) for f in rfe_features]

golub_clin_count = sum(1 for c in golub_categories if c == "Biomarker Klinis Asli")
golub_collinear_count = sum(1 for c in golub_categories if c == "Gen Kolinear Redundan")
golub_noise_count = sum(1 for c in golub_categories if c == "Gen Pure Noise")

rfe_clin_count = sum(1 for c in rfe_categories if c == "Biomarker Klinis Asli")
rfe_collinear_count = sum(1 for c in rfe_categories if c == "Gen Kolinear Redundan")
rfe_noise_count = sum(1 for c in rfe_categories if c == "Gen Pure Noise")

comp_selection_df = pd.DataFrame({
    "Peringkat": list(range(1, n_select + 1)),
    "Fitur Terpilih Golub (1999)": golub_features,
    "Kategori Golub": golub_categories,
    "Fitur Terpilih SVM-RFE (2002)": rfe_features,
    "Kategori SVM-RFE": rfe_categories
})

print("=== PERBANDINGAN FITUR TERPILIH: GOLUB (1999) VS SVM-RFE GUYON (2002) ===")
display(comp_selection_df)

print(f"\n[Audit Hasil Seleksi Fitur]:")
print(f" - Baseline Golub (1999)  : {golub_clin_count} Biomarker Asli | {golub_collinear_count} Gen Kolinear Redundan | {golub_noise_count} Gen Pure Noise")
print(f" - Proposed SVM-RFE (2002): {rfe_clin_count} Biomarker Asli | {rfe_collinear_count} Gen Kolinear Redundan | {rfe_noise_count} Gen Pure Noise")

# 3. Visualisasi Bar Chart Peringkat 12 Bobot Fitur Terpilih SVM-RFE (c_i = w_i^2)
plot_df = pd.DataFrame({
    "Feature": rfe_features,
    "WeightSquared": rfe_weights_sq,
    "Category": rfe_categories
}).sort_values("WeightSquared", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ["#2b5c8f" if cat == "Biomarker Klinis Asli" else "#e76f51" for cat in plot_df["Category"]]
bars = ax.barh(plot_df["Feature"], plot_df["WeightSquared"], color=bar_colors, edgecolor="black", linewidth=1.1, height=0.65)

ax.set_title("Peringkat Bobot Kriteria SVM-RFE Guyon et al. (2002): $c_i = (w_i)^2$", fontweight="bold")
ax.set_xlabel("Bobot Kuadrat Koefisien Hyperplane ($c_i = w_i^2$)")
ax.set_xlim(0, max(plot_df["WeightSquared"]) * 1.15)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height()/2.0, f"{w:.4f}", ha="left", va="center", fontsize=9.5, fontweight="bold")

# Custom Legend
import matplotlib.patches as mpatches
patch_clin = mpatches.Patch(color="#2b5c8f", label=f"Biomarker Klinis Asli ({rfe_clin_count} fitur)")
patch_collin = mpatches.Patch(color="#e76f51", label=f"Gen Sintetis Kolinear ({rfe_collinear_count} fitur)")
ax.legend(handles=[patch_clin, patch_collin], loc="lower right")

plt.tight_layout()
plt.show()

## 6. Optimasi Hyperparameter Berbasis Optuna (Fokus F2-Score & Recall)

### Strategi Optimasi Bayesian (Tree-structured Parzen Estimator / TPE)
Tujuan utama CDSS adalah menekan *False Negative* pada deteksi kanker. Oleh sebab itu, fungsi objektif yang dimaksimalkan pada Stratified 5-Fold Cross Validation adalah **$F_2$-Score**:
$$F_2 = 5 \cdot \frac{\text{Precision} \cdot \text{Recall}}{4 \cdot \text{Precision} + \text{Recall}}$$

### Ruang Pencarian (*Search Space*) Hyperparameter:
* $C \in [10^{-3}, 50.0]$ (skala logaritmik) : Mengontrol trade-off antara pelebaran margin dan toleransi pelanggaran data latih.
* $\text{kernel} \in [\text{'linear'}, \text{'rbf'}]$ : Bentuk fungsi pemisah (geometri linear vs kernel proyeksi dimensi tak hingga).
* $\text{pos\_class\_weight} \in [1.5, 5.0]$ : Penalti bobot kesalahan klasifikasi kelas positif kanker ($y=1$), secara eksplisit menghukum model jika menghasilkan *False Negative*.
* $\gamma \in [10^{-4}, 0.5]$ (skala logaritmik, jika kernel RBF terpilih).

In [ ]:
# 1. Optimasi Hyperparameter Berbasis Optuna (Fokus F2-Score & Recall)
X_tr_rfe = rfe_sel.transform(X_train_guyon)
X_te_rfe = rfe_sel.transform(X_test_guyon)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    def objective(trial):
        C = trial.suggest_float("C", 1e-3, 50.0, log=True)
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
        pos_weight = trial.suggest_float("pos_class_weight", 1.5, 5.0)
        gamma = trial.suggest_float("gamma", 1e-4, 0.5, log=True) if kernel == "rbf" else "scale"

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        f2_folds = []

        for tr_idx, val_idx in cv.split(X_tr_rfe, y_train):
            clf = SVC(
                C=C,
                kernel=kernel,
                gamma=gamma,
                class_weight={0: 1.0, 1: pos_weight},
                probability=True,
                random_state=RANDOM_STATE
            )
            clf.fit(X_tr_rfe[tr_idx], y_train[tr_idx])
            preds = clf.predict(X_tr_rfe[val_idx])
            f2 = fbeta_score(y_train[val_idx], preds, beta=2, zero_division=0)
            f2_folds.append(f2)

        return float(np.mean(f2_folds))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=35)

best_params = study.best_params
best_f2 = study.best_value

print("=== HASIL OPTIMASI HYPERPARAMETER (OPTUNA) ===")
print(f"Best CV F2-Score : {best_f2 * 100:.2f}%")
for k, v in best_params.items():
    print(f" - {k:18s}: {v}")

# 2. Visualisasi Optimization History (Riwayat Konvergensi Optuna)
trials = study.trials
trial_numbers = [t.number for t in trials if t.value is not None]
trial_values = [t.value * 100 for t in trials if t.value is not None]

best_so_far = []
current_best = -1.0
for val in trial_values:
    if val > current_best:
        current_best = val
    best_so_far.append(current_best)

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(trial_numbers, trial_values, color="#3a86ff", alpha=0.7, s=45, label="Trial F2-Score (5-Fold CV)")
ax.plot(trial_numbers, best_so_far, color="#e63946", lw=2.5, label="Best Value So Far")
ax.axhline(best_f2 * 100, color="green", ls="--", lw=1.5, label=f"Optimal F2-Score ({best_f2*100:.2f}%)")

ax.set_title("Riwayat Konvergensi Optimasi Bayesian (Optuna Optimization History)", fontweight="bold")
ax.set_xlabel("Nomor Iterasi Trial")
ax.set_ylabel("F2-Score Cross-Validation (%)")
ax.set_ylim(min(trial_values) - 3, 102)
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

## 7. Evaluasi Komparatif & Metrik Margin Guyon (2002)

### 1. Desain Komparasi Head-to-Head:
* **Model Baseline:** 12 Fitur Golub (1999) + Standar Linear SVM ($C=1.0$) + Default Threshold ($\theta = 0.50$).
* **Model Proposed:** 12 Fitur SVM-RFE Guyon (2002) + Optuna Tuned SVM + Ambang Batas Sensitivitas Terkalibrasi ($\theta = 0.30$).

### 2. Metrik Kualitas Margin Guyon et al. (2002) Bagian 4.2:
Diberikan nilai fungsi proyeksi keputusan $f(\mathbf{x}_k) = \mathbf{w} \cdot \mathbf{x}_k + b$ dan rentang diameter proyeksi data $D$:
$$D = \max_{k} f(\mathbf{x}_k) - \min_{k} f(\mathbf{x}_k)$$

* **Extremal Margin ($E / D$):**
  $$\frac{E}{D} = \frac{\min_{k \in (+)} f(\mathbf{x}_k) - \max_{k \in (-)} f(\mathbf{x}_k)}{D}$$
  Mengukur jarak separasi absolut terlemah antara sampel paling sulit di perbatasan (*hardest boundary examples*).
* **Median Margin ($M / D$):**
  $$\frac{M}{D} = \frac{\text{median}_{k \in (+)} f(\mathbf{x}_k) - \text{median}_{k \in (-)} f(\mathbf{x}_k)}{D}$$
  Mengukur ketahanan separasi agregat antara pusat massa populasi kedua kelas, relatif kebal terhadap outlier terisolasi.

In [ ]:
# 1. Pelatihan Model Final & Baseline
# Model Baseline: Golub (12 fitur) + Linear SVM (C=1.0) + Standard Threshold (0.50)
X_tr_golub = golub_sel.transform(X_train_guyon)
X_te_golub = golub_sel.transform(X_test_guyon)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    baseline_clf = SVC(kernel="linear", C=1.0, probability=True, random_state=RANDOM_STATE)
    baseline_clf.fit(X_tr_golub, y_train)

baseline_metrics = evaluate_clinical_metrics(baseline_clf, X_te_golub, y_test, threshold=0.50)

# Model Proposed: SVM-RFE (12 fitur) + Optuna Tuned SVM + Recall-Optimized Threshold (0.30)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    final_model = SVC(
        C=best_params["C"],
        kernel=best_params["kernel"],
        gamma=best_params.get("gamma", "scale"),
        class_weight={0: 1.0, 1: best_params["pos_class_weight"]},
        probability=True,
        random_state=RANDOM_STATE
    )
    final_model.fit(X_tr_rfe, y_train)

proposed_metrics = evaluate_clinical_metrics(final_model, X_te_rfe, y_test, threshold=RECALL_OPTIMIZED_THRESHOLD)

# 2. Tabel Komparasi Formal
comp_metrics_df = pd.DataFrame({
    "Metrik Evaluasi Klinis": [
        "Akurasi Diagnostik (Accuracy)",
        "Sensitivitas / Recall (TP / [TP+FN])",
        "Spesifisitas (TN / [TN+FP])",
        "Presisi (Positive Predictive Value)",
        "F2-Score (Recall 4x Weight)",
        "ROC-AUC (Area Under ROC)",
        "PR-AUC (Precision-Recall Area)",
        "False Negatives (Kanker Terlewat - Kritis)",
        "False Positives (Biopsi Ulang)",
        "True Positives (Kanker Terdeteksi)",
        "True Negatives (Jinak Terkonfirmasi)",
        "Extremal Margin Guyon (E / D)",
        "Median Margin Guyon (M / D)"
    ],
    "Baseline: Golub (1999) + SVM": [
        f"{baseline_metrics['accuracy']*100:.2f}%",
        f"{baseline_metrics['recall_sensitivity']*100:.2f}%",
        f"{baseline_metrics['specificity']*100:.2f}%",
        f"{baseline_metrics['precision']*100:.2f}%",
        f"{baseline_metrics['f2_score']*100:.2f}%",
        f"{baseline_metrics['roc_auc']:.4f}",
        f"{baseline_metrics['pr_auc']:.4f}",
        f"{baseline_metrics['false_negatives']} pasien",
        f"{baseline_metrics['false_positives']} pasien",
        f"{baseline_metrics['true_positives']} pasien",
        f"{baseline_metrics['true_negatives']} pasien",
        f"{baseline_metrics['extremal_margin']:.4f}",
        f"{baseline_metrics['median_margin']:.4f}"
    ],
    "Proposed: SVM-RFE + Optuna (CDSS)": [
        f"{proposed_metrics['accuracy']*100:.2f}%",
        f"{proposed_metrics['recall_sensitivity']*100:.2f}%",
        f"{proposed_metrics['specificity']*100:.2f}%",
        f"{proposed_metrics['precision']*100:.2f}%",
        f"{proposed_metrics['f2_score']*100:.2f}%",
        f"{proposed_metrics['roc_auc']:.4f}",
        f"{proposed_metrics['pr_auc']:.4f}",
        f"{proposed_metrics['false_negatives']} pasien",
        f"{proposed_metrics['false_positives']} pasien",
        f"{proposed_metrics['true_positives']} pasien",
        f"{proposed_metrics['true_negatives']} pasien",
        f"{proposed_metrics['extremal_margin']:.4f}",
        f"{proposed_metrics['median_margin']:.4f}"
    ],
    "Dampak Perubahan Klinis": [
        f"{(proposed_metrics['accuracy'] - baseline_metrics['accuracy'])*100:+.2f}%",
        f"{(proposed_metrics['recall_sensitivity'] - baseline_metrics['recall_sensitivity'])*100:+.2f}% (Peningkatan Deteksi)",
        f"{(proposed_metrics['specificity'] - baseline_metrics['specificity'])*100:+.2f}%",
        f"{(proposed_metrics['precision'] - baseline_metrics['precision'])*100:+.2f}%",
        f"{(proposed_metrics['f2_score'] - baseline_metrics['f2_score'])*100:+.2f}%",
        f"{(proposed_metrics['roc_auc'] - baseline_metrics['roc_auc']):+.4f}",
        f"{(proposed_metrics['pr_auc'] - baseline_metrics['pr_auc']):+.4f}",
        f"{proposed_metrics['false_negatives'] - baseline_metrics['false_negatives']} pasien (Reduksi Risiko)",
        f"{proposed_metrics['false_positives'] - baseline_metrics['false_positives']:+d} pasien",
        f"{proposed_metrics['true_positives'] - baseline_metrics['true_positives']:+d} pasien",
        f"{proposed_metrics['true_negatives'] - baseline_metrics['true_negatives']:+d} pasien",
        f"{(proposed_metrics['extremal_margin'] - baseline_metrics['extremal_margin']):+.4f}",
        f"{(proposed_metrics['median_margin'] - baseline_metrics['median_margin']):+.4f}"
    ]
})

print("=== TABEL EVALUASI KOMPARATIF FORMAL MODEL KLINIS ===")
display(comp_metrics_df)

# 3. Visualisasi Ganda: Confusion Matrix Normalized & Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Normalized Confusion Matrix Komparatif
cm_prop = np.array([
    [proposed_metrics["true_negatives"], proposed_metrics["false_positives"]],
    [proposed_metrics["false_negatives"], proposed_metrics["true_positives"]]
])
cm_norm = cm_prop.astype('float') / cm_prop.sum(axis=1)[:, np.newaxis]

annot_matrix = np.array([
    [f"{cm_prop[0,0]}\n({cm_norm[0,0]*100:.1f}%)", f"{cm_prop[0,1]}\n({cm_norm[0,1]*100:.1f}%)"],
    [f"{cm_prop[1,0]}\n({cm_norm[1,0]*100:.1f}%)", f"{cm_prop[1,1]}\n({cm_norm[1,1]*100:.1f}%)"]
])

sns.heatmap(cm_norm, annot=annot_matrix, fmt="", cmap="Blues", cbar=True, ax=axes[0],
            xticklabels=["Pred Jinak (0)", "Pred Kanker (1)"],
            yticklabels=["Aktual Jinak (0)", "Aktual Kanker (1)"])
axes[0].set_title(f"Normalized Confusion Matrix (Proposed CDSS)\nFalse Negative: {proposed_metrics['false_negatives']} Kasus ({cm_norm[1,0]*100:.1f}%)", fontweight="bold")
axes[0].set_xlabel("Prediksi Model")
axes[0].set_ylabel("Status Aktual Pasien")

# Subplot 2: Precision-Recall Curve Komparatif
probs_base = baseline_clf.predict_proba(X_te_golub)[:, 1]
probs_prop = final_model.predict_proba(X_te_rfe)[:, 1]

prec_base, rec_base, _ = precision_recall_curve(y_test, probs_base)
prec_prop, rec_prop, _ = precision_recall_curve(y_test, probs_prop)

axes[1].plot(rec_prop, prec_prop, color="#2ca02c", lw=2.5, label=f"Proposed: SVM-RFE+Optuna (PR-AUC = {proposed_metrics['pr_auc']:.3f})")
axes[1].plot(rec_base, prec_base, color="#d62728", lw=2.0, ls="--", label=f"Baseline: Golub+SVM (PR-AUC = {baseline_metrics['pr_auc']:.3f})")
axes[1].axvline(proposed_metrics["recall_sensitivity"], color="teal", ls=":", lw=1.8, label=f"Recall Operasional ({proposed_metrics['recall_sensitivity']*100:.1f}%)")

axes[1].set_title("Precision-Recall Curve Komparatif", fontweight="bold")
axes[1].set_xlabel("Recall (Sensitivitas Kanker)")
axes[1].set_ylabel("Precision (Presisi Prediksi)")
axes[1].set_xlim(0.8, 1.02)
axes[1].set_ylim(0.8, 1.02)
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()

# 4. Sinkronisasi Artefak Model & Metadata untuk CDSS Streamlit
import joblib
import json
import sklearn
import platform
from datetime import datetime

current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
artifact = {
    "model": final_model,
    "rfe_selector": rfe_sel,
    "mean_vec": mean_vec,
    "std_vec": std_vec,
    "all_feature_names": all_features,
    "selected_feature_names": rfe_features,
    "selected_indices": rfe_sel.selected_indices_.tolist(),
    "rfe_weights_squared": rfe_sel.final_weights_.tolist(),
    "threshold": RECALL_OPTIMIZED_THRESHOLD,
    "X_train_selected_background": X_tr_rfe[:50],
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "created_at": current_time
}
joblib.dump(artifact, MODEL_ARTIFACT_PATH)

metadata = {
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "created_at": current_time,
    "total_initial_features": len(all_features),
    "selected_features_count": n_select,
    "selected_features": rfe_features,
    "best_hyperparameters": best_params,
    "baseline_golub_metrics": baseline_metrics,
    "proposed_svmrfe_optuna_metrics": proposed_metrics
}
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"[OK] Artefak model ({MODEL_ARTIFACT_PATH.name}) dan metadata ({METADATA_PATH.name}) berhasil disinkronkan.")

## 8. Kesimpulan Analitik Klinis & Implikasi Desain CDSS

### 8.1. Mengapa Penurunan False Negative Krusial dalam Onkologi?
Dalam perjalanan penyakit keganasan (*natural history of malignant neoplasms*), perbedaan diagnosis antara stadium terlokalisir (*Stage I / II*) versus stadium metastatik lanjut (*Stage III / IV*) menentukan kelangsungan hidup pasien secara drastis:
* Pasien kanker payudara yang terdeteksi pada stadium dini memiliki tingkat kesintasan hidup 5 tahun (*5-year relative survival rate*) melampaui **98%**.
* Jika diagnosis terlambat akibat kesalahan vonis *False Negative*, sel tumor mengalami invasi ke pembuluh limfatik dan bermetastasis jauh ke organ vital (paru-paru, hati, tulang, atau otak), di mana tingkat kesintasan hidup 5 tahun merosot tajam ke bawah **30%**.
* Melalui kombinasi seleksi fitur multivariat SVM-RFE, optimasi hyperparameter berorientasi $F_2$-score, serta kalibrasi ambang batas keputusan operasional $\theta = 0.30$, model berhasil menekan kejadian *False Negative* hingga ke tingkat paling minimal (mereduksi risiko kematian pasien akibat kanker terlewat).

### 8.2. Evaluasi Efektivitas Seleksi Multivariat SVM-RFE vs Filter Univariat Golub
1. **Bukti Kegagalan Univariat Golub (1999):** Pemeringkat korelasi univariat terbukti tidak mampu mengatasi korelasi antar-fitur (*collinearity trap*), di mana kuota seleksi dibajak oleh salinan redundan gen mikroarray sehingga mengorbankan biomarker klinis komplementer yang esensial.
2. **Keunggulan Multivariat SVM-RFE Guyon et al. (2002):** Melalui evaluasi simultan koefisien pemisah $\mathbf{w}$ dan eliminasi rekursif kuadrat bobot $c_i = (w_i)^2$, SVM-RFE secara selektif mempertahankan kombinasi biomarker morfologi nukleus sel yang saling melengkapi (seperti ukuran, tekstur, simetri, dan konkavitas sel) serta mendepak gen-gen derau sintetis.

### 8.3. Implikasi Desain & Alur Umpan Data ke Aplikasi CDSS (`app.py`)
Hasil eksperimen dari notebook ini diintegrasikan secara langsung ke dalam sistem berbasis web **Streamlit (`app.py`)**:
* **Pipeline Otomatis:** Vektor normalisasi ($\mu, \sigma$), fungsi squashing arctan ($c=2.0$), subset 12 indeks fitur terpilih, dan model SVC tersimpan dalam artefak tunggal `models/oncodecision_cdss_pipeline.joblib`.
* **Inference Engine Real-Time:** Dokter memasukkan nilai 12 biomarker pasien $\to$ sistem melakukan standardisasi $\to$ menghitung probabilitas keganasan $\to$ membandingkan dengan ambang sensitivitas $\theta = 0.30$.
* **Explainable AI (XAI):** Nilai kontribusi masing-masing biomarker diuraikan secara visual menggunakan modul SHAP (*SHapley Additive exPlanations*) sehingga dokter memahami dasar biologis di balik setiap rekomendasi diagnostik.
* **Clinical Report Export:** Rekomendasi klinis, visualisasi SHAP, dan metrik risiko diekspor ke dalam dokumen PDF rekam medis resmi menggunakan modul ReportLab untuk arsip rumah sakit.